# Combine Amenities: Spreadsheet A + Spreadsheet B

This notebook:
1. Loads two spreadsheets
2. Matches rows by Costar ID (or address as fallback)
3. Merges amenities from both — no duplicates
4. Saves the result back into Spreadsheet A's format

## 0. Configuration — Edit This Section

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
SPREADSHEET_A = "PI Whyte Ave.xlsx"            # Input/output — supports .xlsx, .xls, .csv
SPREADSHEET_B = "Costar Office Dataset.xlsx"   # Source of additional amenities
OUTPUT_PATH   = "PI Whyte Ave updated.xlsx"    # Where to save the result

# ── Column names in Spreadsheet A ────────────────────────────────────────────
A_ID_COL        = "Costar ID"       # Primary match key — set to None to skip
A_ADDRESS_COL   = "Primary address" # Fallback match key
A_AMENITIES_COL = "Amenities"

# ── Column names in Spreadsheet B ────────────────────────────────────────────
B_ID_COL        = "Costar ID"        # Primary match key — set to None to skip
B_ADDRESS_COL   = "Property Address"
B_AMENITIES_COL = "Amenities"

# ── Delimiter used inside amenity cells ──────────────────────────────────────
DELIMITER        = ","   # How amenities are separated in the source files
OUTPUT_DELIMITER = ", "  # How amenities are separated in the output

## 1. Install & Import Dependencies

In [ ]:
%pip install pandas openpyxl --quiet
import os
import pandas as pd
print("Ready.")

## 2. Load the Spreadsheets

In [ ]:
def load_file(path):
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".csv":
        return pd.read_csv(path, dtype=str)
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path, dtype=str)
    raise ValueError(f"Unsupported file type: {ext}")

df_a = load_file(SPREADSHEET_A)
df_b = load_file(SPREADSHEET_B)

print(f"Spreadsheet A: {len(df_a)} rows  |  columns: {df_a.columns.tolist()}")
print(f"Spreadsheet B: {len(df_b)} rows  |  columns: {df_b.columns.tolist()}")

## 3. Helper Functions

In [ ]:
def split_amenities(raw):
    """Split a delimited amenities cell into a list of trimmed strings."""
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [a.strip() for a in str(raw).split(DELIMITER) if a.strip()]


def merge_amenities(list_a, list_b):
    """Combine two amenity lists. Duplicates removed (case-insensitive). Order preserved, A first."""
    seen, merged = set(), []
    for item in list_a + list_b:
        key = item.lower()
        if key not in seen:
            seen.add(key)
            merged.append(item)
    return merged


def normalise_address(addr):
    """Lowercase and collapse whitespace for reliable address comparison."""
    if pd.isna(addr):
        return ""
    return " ".join(str(addr).strip().lower().split())


print("Helper functions loaded.")

## 4. Build Lookup Tables from Spreadsheet B

In [ ]:
id_lookup   = {}  # Costar ID  → amenities list
addr_lookup = {}  # address    → amenities list

has_b_id = B_ID_COL and B_ID_COL in df_b.columns

for _, row in df_b.iterrows():
    amenities = split_amenities(row[B_AMENITIES_COL])

    if has_b_id and not pd.isna(row.get(B_ID_COL, float('nan'))):
        bid = str(row[B_ID_COL]).strip()
        if bid:
            id_lookup[bid] = merge_amenities(id_lookup.get(bid, []), amenities)

    addr = normalise_address(row[B_ADDRESS_COL])
    if addr:
        addr_lookup[addr] = merge_amenities(addr_lookup.get(addr, []), amenities)

print(f"ID lookup   : {len(id_lookup):,} unique Costar IDs")
print(f"Addr lookup : {len(addr_lookup):,} unique addresses")

## 5. Merge Amenities into Spreadsheet A

In [ ]:
df_out = df_a.copy()
has_a_id = A_ID_COL and A_ID_COL in df_out.columns
updated = 0

for idx, row in df_out.iterrows():
    b_amenities = None

    # 1. Match by Costar ID
    if has_a_id and not pd.isna(row.get(A_ID_COL, float('nan'))):
        aid = str(row[A_ID_COL]).strip()
        if aid in id_lookup:
            b_amenities = id_lookup[aid]

    # 2. Match by address (fallback)
    if b_amenities is None:
        addr = normalise_address(row[A_ADDRESS_COL])
        if addr in addr_lookup:
            b_amenities = addr_lookup[addr]

    if b_amenities is None:
        continue

    a_amenities = split_amenities(row[A_AMENITIES_COL])
    merged = merge_amenities(a_amenities, b_amenities)

    if merged != a_amenities:
        df_out.at[idx, A_AMENITIES_COL] = OUTPUT_DELIMITER.join(merged)
        updated += 1

print(f"Rows updated: {updated}")

## 6. Review Changes

In [ ]:
changed = df_out[A_AMENITIES_COL].fillna("") != df_a[A_AMENITIES_COL].fillna("")

comparison = pd.DataFrame({
    A_ADDRESS_COL        : df_a.loc[changed, A_ADDRESS_COL],
    "amenities_before"   : df_a.loc[changed, A_AMENITIES_COL],
    "amenities_after"    : df_out.loc[changed, A_AMENITIES_COL],
})

print(f"{len(comparison)} row(s) changed.")
display(comparison.reset_index(drop=True))

## 7. Save Output

In [ ]:
ext = os.path.splitext(OUTPUT_PATH)[-1].lower()
if ext == ".csv":
    df_out.to_csv(OUTPUT_PATH, index=False)
else:
    df_out.to_excel(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Shape: {df_out.shape[0]} rows × {df_out.shape[1]} columns")